# MVP v1: расчетные KPI и сравнение методологий

Цель:
- на том же периметре `ИНН + agr_id` рассчитать core + расчетные KPI;
- сравнить две методологии:
  1. `DAG_techlead` (что можно посчитать строго по подтвержденным входам DAG),
  2. `Business_formula` (канонические поля отчета Excel).

Примечание:
- для метрик, где в DAG нет подтвержденных входов, выводится `NULL` и причина в `input_gaps_registry`.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
top_n = 10

# Допуски сравнения
eps_count = 0.0
eps_amount = 0.01
eps_pct = 0.01

# Бизнес-константа
aur_per_terminal = 1926.0

# Канонические поля отчета
excel_inn_col = 'ИНН'
excel_agr_id_col = 'ID договора'
excel_name_col = 'Наименование'
excel_contract_col = 'Номер договора'

report_columns_canonical = [
    'Филиал', 'Номер организации', 'ID договора', 'Наименование', 'ИНН',
    'Номер договора', 'Дата регистрации договора', 'Дата закрытия договора',
    'Количество дней', 'Договоры менее 90 дней', 'ВСП договора', 'Код ВСП',
    'Тариф', 'Отношение к ГК', 'Принадлежность терминального оборудования',
    'Ко-во торговых точек', 'Кол-во терминалов', 'АУР', 'Амортизация',
    'Количеств операций', 'Сумма операций', 'Комиссия \n(% с операций)',
    'Комиссия \n(₽ в месяц)', 'Комиссия эквайринга',
    'Комиссия МПС (IRF, ₽)', 'ЧОД', 'Средний экв. %', 'Средний IRF %',
    'Фин. Рез.', 'Всего ТСП', 'Эффективные ТСП', 'Неэффективные ТСП',
    'Учитывать в расчетах'
]

business_metric_cols = {
    'trx_cnt': 'Количеств операций',
    'trx_sum': 'Сумма операций',
    'retl_cnt': 'Ко-во торговых точек',
    'term_cnt': 'Кол-во терминалов',
    'aur': 'АУР',
    'amortization': 'Амортизация',
    'commission_from_ops': 'Комиссия \n(% с операций)',
    'commission_monthly': 'Комиссия \n(₽ в месяц)',
    'commission_total': 'Комиссия эквайринга',
    'commission_irf': 'Комиссия МПС (IRF, ₽)',
    'nod': 'ЧОД',
    'avg_acq_pct': 'Средний экв. %',
    'avg_irf_pct': 'Средний IRF %',
    'fin_result': 'Фин. Рез.',
}

print('month_start =', month_start)
print('month_end =', month_end)
print('top_n =', top_n)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_pos_terminals')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
print('Impala initialized, metadata invalidated')

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if s == '':
        return None
    return s

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_agr_id(v):
    return normalize_numeric_str(v)

def normalize_text(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def normalize_num(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s == '' or s.lower() in {'nan', 'none', 'null', '-'}:
        return None
    s = s.replace(',', '.')
    try:
        return float(Decimal(s))
    except Exception:
        return pd.to_numeric(s, errors='coerce')

def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = {str(c).strip() for c in required_cols}
    for i in range(min(scan_rows, len(raw))):
        vals = {str(x).strip() for x in raw.iloc[i].tolist()}
        if len(req & vals) >= 2:
            return i
    return 0

def to_sql_in_list(vals):
    safe = [str(v).replace("'", "''") for v in vals if pd.notna(v)]
    return ','.join([f"'{x}'" for x in safe])

## 1) Периметр `top10_common_keys` (тот же подход, что и в первой тетрадке)

In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_agr_id_col, excel_contract_col, excel_name_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

missing_required = [c for c in [excel_inn_col, excel_agr_id_col, excel_contract_col, excel_name_col] if c not in excel_raw.columns]
if missing_required:
    raise ValueError(f'Missing required columns in Excel: {missing_required}')

missing_canonical = [c for c in report_columns_canonical if c not in excel_raw.columns]
extra_columns = [c for c in excel_raw.columns if c not in report_columns_canonical]
print('Missing canonical columns:', missing_canonical)
print('Extra columns vs canonical:', extra_columns)

excel_df = excel_raw.copy()
excel_df['inn_norm'] = excel_df[excel_inn_col].apply(normalize_inn)
excel_df['agr_id_norm'] = excel_df[excel_agr_id_col].apply(normalize_agr_id)
excel_df['excel_name_norm'] = excel_df[excel_name_col].apply(normalize_text)
excel_df['excel_contract_norm'] = excel_df[excel_contract_col].apply(normalize_text)

excel_keys = (
    excel_df[['inn_norm', 'agr_id_norm']]
    .dropna(subset=['inn_norm', 'agr_id_norm'])
    .drop_duplicates()
    .reset_index(drop=True)
)
print('Excel keys:', len(excel_keys))

In [ ]:
sql_lake_keys = f"""
with sa_active as (
    select
        regexp_replace(trim(c.c_inn), '[^0-9]', '') as inn_raw,
        cast(a.abs_agr_id as string) as agr_id_norm
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
      and a.abs_agr_id is not null
), cleaned as (
    select
        case
          when length(inn_raw) = 9 then lpad(inn_raw, 10, '0')
          when length(inn_raw) = 11 then lpad(inn_raw, 12, '0')
          else inn_raw
        end as inn_norm,
        agr_id_norm
    from sa_active
)
select distinct inn_norm, agr_id_norm
from cleaned
where inn_norm is not null and inn_norm <> ''
"""

with imp:
    lake_keys = imp.fetch(sql_lake_keys)

lake_keys['inn_norm'] = lake_keys['inn_norm'].apply(normalize_inn)
lake_keys['agr_id_norm'] = lake_keys['agr_id_norm'].apply(normalize_agr_id)
lake_keys = lake_keys.dropna(subset=['inn_norm', 'agr_id_norm']).drop_duplicates()

top10_common_keys = (
    excel_keys
    .merge(lake_keys, on=['inn_norm', 'agr_id_norm'], how='inner')
    .sort_values(['inn_norm', 'agr_id_norm'])
    .head(top_n)
    .reset_index(drop=True)
)

print('Lake keys:', len(lake_keys))
print('Top common keys:', len(top10_common_keys))
display(top10_common_keys)

## 2) DAG-техлид слой: core + расчетные KPI (с реестром пробелов входов)

In [ ]:
if top10_common_keys.empty:
    raise ValueError('top10_common_keys is empty')

sample_rows_sql = '\n    union all\n    '.join([
    f"select '{r.inn_norm}' as inn_norm, '{r.agr_id_norm}' as agr_id_norm"
    for r in top10_common_keys.itertuples(index=False)
])

sql_dag_metrics = f"""
with sample_keys as (
    {sample_rows_sql}
),
base as (
    select
        sk.inn_norm,
        sk.agr_id_norm,
        a.n_agr,
        a.n_cmp_client,
        c.c_cmp_name as client_name,
        a.c_agr_number as contract_number
    from sample_keys sk
    join ods_alpha.scd1_agreements a
      on cast(a.abs_agr_id as string) = sk.agr_id_norm
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
),
retl_by_key as (
    select
        b.inn_norm,
        b.agr_id_norm,
        count(distinct cast(m.c_nmrc as string)) as retl_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and coalesce(m.ods_deleted_flg, '0') <> '1'
     and m.c_nmrc is not null
    group by b.inn_norm, b.agr_id_norm
),
term_by_key as (
    select
        b.inn_norm,
        b.agr_id_norm,
        count(distinct cast(t.c_nter as string)) as term_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and coalesce(m.ods_deleted_flg, '0') <> '1'
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and coalesce(t.ods_deleted_flg, '0') <> '1'
     and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
     and (t.d_ter_close is null or cast(t.d_ter_close as date) >= cast('{month_start}' as date))
    group by b.inn_norm, b.agr_id_norm
),
trx_filtered as (
    select
        t.n_trx,
        cast(t.n_amt_src as double) as n_amt_src
    from ods_alpha.scd1_trx t
    where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
      and t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and coalesce(t.cf_trx_stat, '') <> 'R'
      and coalesce(t.ods_deleted_flg, '0') <> '1'
),
trx_by_key as (
    select
        b.inn_norm,
        b.agr_id_norm,
        count(distinct tf.n_trx) as trx_cnt,
        sum(tf.n_amt_src) as trx_sum,
        cast(null as double) as commission_from_ops
    from base b
    left join ods_alpha.scd1_trx_acq ta
      on ta.n_agr = b.n_agr
     and coalesce(ta.ods_deleted_flg, '0') <> '1'
    left join trx_filtered tf
      on tf.n_trx = ta.n_trx
    group by b.inn_norm, b.agr_id_norm
)
select
    b.inn_norm,
    b.agr_id_norm,
    b.client_name,
    b.contract_number,
    coalesce(trx.trx_cnt, 0) as trx_cnt,
    coalesce(trx.trx_sum, 0) as trx_sum,
    coalesce(r.retl_cnt, 0) as retl_cnt,
    coalesce(t.term_cnt, 0) as term_cnt,
    coalesce(trx.commission_from_ops, 0) as commission_from_ops
from base b
left join retl_by_key r on r.inn_norm = b.inn_norm and r.agr_id_norm = b.agr_id_norm
left join term_by_key t on t.inn_norm = b.inn_norm and t.agr_id_norm = b.agr_id_norm
left join trx_by_key trx on trx.inn_norm = b.inn_norm and trx.agr_id_norm = b.agr_id_norm
order by b.inn_norm, b.agr_id_norm
"""

with imp:
    dag_metrics_top10 = imp.fetch(sql_dag_metrics)

dag_metrics_top10

In [ ]:
# Расчетные KPI DAG-методологии + реестр пробелов входов
dag_metrics_top10 = dag_metrics_top10.copy()

dag_metrics_top10['aur'] = dag_metrics_top10['term_cnt'].fillna(0) * aur_per_terminal
dag_metrics_top10['amortization'] = None  # нет подтвержденного входа в DAG
dag_metrics_top10['commission_monthly'] = None  # нет подтвержденного входа в DAG
dag_metrics_top10['commission_total'] = None
dag_metrics_top10['commission_irf'] = None  # нет подтвержденного входа в DAG
dag_metrics_top10['nod'] = None  # формула ЧОД не подтверждена полностью
dag_metrics_top10['avg_acq_pct'] = dag_metrics_top10.apply(
    lambda r: (r['commission_from_ops'] / r['trx_sum'] * 100.0) if pd.notna(r['trx_sum']) and r['trx_sum'] not in (0, 0.0) else None,
    axis=1
)
dag_metrics_top10['avg_irf_pct'] = None  # нет подтвержденного IRF входа
dag_metrics_top10['fin_result'] = None  # нет полного набора входов (ЧОД, амортизация)

# Заполняем commission_total только если есть подтвержденные входы обеих частей
dag_metrics_top10['commission_total'] = dag_metrics_top10.apply(
    lambda r: (r['commission_from_ops'] + r['commission_monthly'])
    if pd.notna(r['commission_from_ops']) and pd.notna(r['commission_monthly']) else None,
    axis=1
)

dag_metrics_top10['methodology'] = 'DAG_techlead'

display(dag_metrics_top10)

In [ ]:
input_gaps_registry = pd.DataFrame([
    {
        'metric_name': 'amortization',
        'is_fully_dag_based': False,
        'input_gap_reason': 'Амортизация по текущей методике берется из внешнего Excel-источника.'
    },
    {
        'metric_name': 'commission_monthly',
        'is_fully_dag_based': False,
        'input_gap_reason': 'Фиксированная комиссия (руб./месяц) в DAG-слое не подтверждена единым расчетным входом.'
    },
    {
        'metric_name': 'commission_irf',
        'is_fully_dag_based': False,
        'input_gap_reason': 'Комиссия МПС (IRF) не выделена подтвержденным полем в текущем контуре DAG.'
    },
    {
        'metric_name': 'nod',
        'is_fully_dag_based': False,
        'input_gap_reason': 'Полная формула ЧОД требует дополнительных компонентов/правил, не финализированных в DAG.'
    },
    {
        'metric_name': 'fin_result',
        'is_fully_dag_based': False,
        'input_gap_reason': 'Фин. результат зависит от ЧОД и амортизации, которые не полностью подтверждены в DAG-слое.'
    },
    {
        'metric_name': 'trx_cnt',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
    {
        'metric_name': 'trx_sum',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
    {
        'metric_name': 'retl_cnt',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
    {
        'metric_name': 'term_cnt',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
    {
        'metric_name': 'aur',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
    {
        'metric_name': 'commission_from_ops',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
    {
        'metric_name': 'avg_acq_pct',
        'is_fully_dag_based': True,
        'input_gap_reason': None
    },
])

display(input_gaps_registry)

## 3) Business-слой по каноническим колонкам Excel

In [ ]:
missing_business_cols = [c for c in business_metric_cols.values() if c not in excel_raw.columns]
if missing_business_cols:
    raise ValueError(f'Missing required business metric columns in Excel: {missing_business_cols}')

business_df = excel_raw.copy()
business_df['inn_norm'] = business_df[excel_inn_col].apply(normalize_inn)
business_df['agr_id_norm'] = business_df[excel_agr_id_col].apply(normalize_agr_id)
business_df['excel_name_norm'] = business_df[excel_name_col].apply(normalize_text)
business_df['excel_contract_norm'] = business_df[excel_contract_col].apply(normalize_text)

for target_metric, source_col in business_metric_cols.items():
    business_df[target_metric] = business_df[source_col].apply(normalize_num)

business_metrics_top10 = (
    business_df
    .dropna(subset=['inn_norm', 'agr_id_norm'])
    .merge(top10_common_keys, on=['inn_norm', 'agr_id_norm'], how='inner')
    .groupby(['inn_norm', 'agr_id_norm'], as_index=False)
    .agg({
        'excel_name_norm': 'max',
        'excel_contract_norm': 'max',
        'trx_cnt': 'sum',
        'trx_sum': 'sum',
        'retl_cnt': 'max',
        'term_cnt': 'max',
        'aur': 'max',
        'amortization': 'sum',
        'commission_from_ops': 'sum',
        'commission_monthly': 'sum',
        'commission_total': 'sum',
        'commission_irf': 'sum',
        'nod': 'sum',
        'avg_acq_pct': 'max',
        'avg_irf_pct': 'max',
        'fin_result': 'sum',
    })
)
business_metrics_top10['methodology'] = 'Business_formula'

display(business_metrics_top10)

## 4) Сравнение методологий: `methodology_compare`, `comparison_summary`, `top_diffs`

In [ ]:
compare_metrics = [
    'trx_cnt', 'trx_sum', 'retl_cnt', 'term_cnt',
    'aur', 'amortization',
    'commission_from_ops', 'commission_monthly', 'commission_total', 'commission_irf',
    'nod', 'avg_acq_pct', 'avg_irf_pct', 'fin_result'
]

cmp_wide = (
    dag_metrics_top10[
        ['inn_norm', 'agr_id_norm', 'client_name', 'contract_number'] + compare_metrics
    ]
    .merge(
        business_metrics_top10[
            ['inn_norm', 'agr_id_norm', 'excel_name_norm', 'excel_contract_norm'] + compare_metrics
        ],
        on=['inn_norm', 'agr_id_norm'],
        how='left',
        suffixes=('_dag', '_business')
    )
)

rows = []
for rec in cmp_wide.to_dict('records'):
    for m in compare_metrics:
        dag_val = rec.get(f'{m}_dag')
        biz_val = rec.get(f'{m}_business')
        if m in {'trx_cnt', 'retl_cnt', 'term_cnt'}:
            eps = eps_count
        elif m in {'avg_acq_pct', 'avg_irf_pct'}:
            eps = eps_pct
        else:
            eps = eps_amount

        if pd.isna(dag_val) and pd.isna(biz_val):
            status = 'both_missing'
            diff = None
            match = None
        elif pd.isna(dag_val) and pd.notna(biz_val):
            status = 'dag_missing'
            diff = None
            match = None
        elif pd.notna(dag_val) and pd.isna(biz_val):
            status = 'business_missing'
            diff = None
            match = None
        else:
            diff = float(biz_val) - float(dag_val)
            match = abs(diff) <= eps
            status = 'compared'

        rows.append({
            'inn_norm': rec['inn_norm'],
            'agr_id_norm': rec['agr_id_norm'],
            'metric_name': m,
            'dag_value': dag_val,
            'business_value': biz_val,
            'diff': diff,
            'match_flag': match,
            'comparison_status': status,
        })

methodology_compare = pd.DataFrame(rows)
display(methodology_compare.head(200))

In [ ]:
comparison_summary = (
    methodology_compare
    .groupby('metric_name', as_index=False)
    .agg(
        total_keys=('metric_name', 'count'),
        compared_cnt=('comparison_status', lambda s: int((s == 'compared').sum())),
        dag_missing_cnt=('comparison_status', lambda s: int((s == 'dag_missing').sum())),
        business_missing_cnt=('comparison_status', lambda s: int((s == 'business_missing').sum())),
        both_missing_cnt=('comparison_status', lambda s: int((s == 'both_missing').sum())),
        match_cnt=('match_flag', lambda s: int((s == True).sum())),
        mismatch_cnt=('match_flag', lambda s: int((s == False).sum())),
    )
)
comparison_summary['coverage_pct'] = comparison_summary.apply(
    lambda r: round(100.0 * r['compared_cnt'] / r['total_keys'], 2) if r['total_keys'] else 0.0,
    axis=1
)

top_diffs = (
    methodology_compare[methodology_compare['comparison_status'] == 'compared']
    .assign(abs_diff=lambda d: d['diff'].abs())
    .sort_values(['metric_name', 'abs_diff'], ascending=[True, False])
    .groupby('metric_name', as_index=False, group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

display(comparison_summary.sort_values('metric_name'))
display(top_diffs)
display(input_gaps_registry.sort_values(['is_fully_dag_based', 'metric_name']))